In [8]:
experiments = ['defects4j/v2/codegen_350M/exp1_defects4j_v2_1_linear_1024_1_BCE',
               'defects4j/v2/codegen_6B/exp1_defects4j_v2_2_linear_1024_1_BCE',
               'defects4j/v2/codegen_16B/exp1_defects4j_v2_3_linear_1024_1_BCE',
               'defects4j/v2/codegen_16B/exp2_defects4j_v2_3_gru_1024_1_BCE',
               'defects4j/v2/Qwen_QwQ_32B/exp3_defects4j_v2_17_linear_1024_1_BCE',
               'defects4j/v2/DeepSeek_R1_Distill_Llama_8B/exp4_defects4j_v2_18_linear_1024_1_BCE',
               'defects4j/v2/codegen_16B/exp5_defects4j_v2_3_linear_1024_1_CC',
               
               'solidity/v2/Qwen_QwQ_32B/exp6_solidity_v2_17_gru_1024_1_BCE',
               'solidity/v2/DeepSeek_R1_Distill_Llama_8B/exp7_solidity_v2_18_gru_1024_1_BCE',
               'solidity/v2/DeepSeek_R1_Distill_Qwen_14B/exp8_solidity_v2_19_gru_1024_1_BCE',
               'solidity/v3/codegen_16B/exp9_solidity_v3_3_gru_1024_1_BCE',
               'solidity/v2/DeepSeek_R1_Distill_Qwen_14B/exp10_solidity_v2_19_gru_1024_2_BCE',
               'solidity/v2/DeepSeek_R1_Distill_Qwen_14B/exp11_solidity_v2_19_gru_1024_2_BCE',
              
              'solidity_detect_1/v1/DeepSeek_R1_Distill_Qwen_14B/exp_detection_solidity_detect_1_v1_19_1_BCE',
               'solidity_detect_1/v1/DeepSeek_R1_Distill_Qwen_14B/exp_detection_solidity_detect_1_v1_19_2_BCE',
               'solidity_detect_3/v1/DeepSeek_R1_Distill_Qwen_14B/exp_detection_solidity_detect_3_v1_19_1_BCE',
               'solidity_detect_3/v1/DeepSeek_R1_Distill_Qwen_14B/exp_detection_solidity_detect_3_v1_19_2_BCE',
               'solidity_detect_15/v1/DeepSeek_R1_Distill_Qwen_14B/exp_detection_solidity_detect_15_v1_19_1_BCE',
               'solidity_detect_15/v1/DeepSeek_R1_Distill_Qwen_14B/exp_detection_solidity_detect_15_v1_19_2_BCE'
              ]

# csv_files_folder = "tensorboard_csvs"
# result_folder = "results_comparison"

csv_files_folder = "new_tensorboard_csvs"
result_folder = "new_results_comparison"

In [10]:
import os
import pandas as pd

for experiment in experiments:
    print(f"\n=======================================================================================\nExperiment: {experiment}")
    results = {'f1_score': [], 'precision': [], 'recall': [], 'top_1': [], 'top_3': [], 'top_5': []}
    for i in range(0, 10):
        csv_directory = f'{os.getcwd()}/{csv_files_folder}/{experiment}/fold_{i}'
        df_f1_score = pd.read_csv(f'{csv_directory}/f1_score_val.csv')
        df_precision = pd.read_csv(f'{csv_directory}/precision_val.csv')
        df_recall = pd.read_csv(f'{csv_directory}/recall_val.csv')
        if not 'detect' in experiment:
            df_top_1 = pd.read_csv(f'{csv_directory}/top_1_rate_val.csv')
            df_top_3 = pd.read_csv(f'{csv_directory}/top_3_rate_val.csv')
            df_top_5 = pd.read_csv(f'{csv_directory}/top_5_rate_val.csv')

        max_f1_score = df_f1_score['value'].max()
        max_f1_index = df_f1_score['value'].idxmax()
    
        step_at_max_f1 = df_f1_score.iloc[max_f1_index]['step']
    
        # Get precision and recall at that step
        precision_at_max_f1 = df_precision[df_precision['step'] == step_at_max_f1]['value'].values
        recall_at_max_f1 = df_recall[df_recall['step'] == step_at_max_f1]['value'].values
        
        precision_val = max(precision_at_max_f1) if len(precision_at_max_f1) > 0 else None
        recall_val = max(recall_at_max_f1) if len(recall_at_max_f1) > 0 else None

        results['f1_score'].append(round(max_f1_score*100, 2))
        results['precision'].append(round(precision_val*100, 2))
        results['recall'].append(round(recall_val*100, 2))

        if not 'detect' in experiment:
            max_top_1 = df_top_1['value'].max()
            max_top_3 = df_top_3['value'].max()
            max_top_5 = df_top_5['value'].max()
            
            results['top_1'].append(round(max_top_1*100, 2))
            results['top_3'].append(round(max_top_3*100, 2))
            results['top_5'].append(round(max_top_5*100, 2))

        print(f"  Fold {i}: Max F1 Score = {round(max_f1_score * 100, 2)} at step {step_at_max_f1}")
        print(f"           Precision = {round(precision_val * 100, 2) if precision_val is not None else 'N/A'}")
        print(f"           Recall = {round(recall_val * 100, 2) if recall_val is not None else 'N/A'}")

        if not 'detect' in experiment:
            print(f"           Top 1 = {round(max_top_1 * 100, 2) if max_top_1 is not None else 'N/A'}")
            print(f"           Top 3 = {round(max_top_3 * 100, 2) if max_top_3 is not None else 'N/A'}")
            print(f"           Top 5 = {round(max_top_5 * 100, 2) if max_top_5 is not None else 'N/A'}")

    rows = []
    for key in list(results.keys()):
        row = [key] + results[key]
        rows.append(row)
    df = pd.DataFrame(rows, columns=['metric', 'fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4', 'fold_5', 'fold_6', 'fold_7', 'fold_8', 'fold_9'])
    os.makedirs(f"{result_folder}", exist_ok=True)
    df.to_csv(f"{result_folder}/{experiment.split("/")[-1]}.csv", index=False)
        


Experiment: defects4j/v2/codegen_350M/exp1_defects4j_v2_1_linear_1024_1_BCE
  Fold 0: Max F1 Score = 12.97 at step 14.0
           Precision = 8.45
           Recall = 27.93
           Top 1 = 16.07
           Top 3 = 32.14
           Top 5 = 34.82
  Fold 1: Max F1 Score = 18.88 at step 36.0
           Precision = 38.31
           Recall = 12.53
           Top 1 = 15.18
           Top 3 = 28.57
           Top 5 = 36.61
  Fold 2: Max F1 Score = 8.05 at step 19.0
           Precision = 6.95
           Recall = 9.57
           Top 1 = 9.82
           Top 3 = 20.54
           Top 5 = 25.0
  Fold 3: Max F1 Score = 19.83 at step 39.0
           Precision = 46.09
           Recall = 12.63
           Top 1 = 13.39
           Top 3 = 22.32
           Top 5 = 29.46
  Fold 4: Max F1 Score = 9.79 at step 24.0
           Precision = 12.99
           Recall = 7.86
           Top 1 = 8.93
           Top 3 = 20.54
           Top 5 = 28.57
  Fold 5: Max F1 Score = 19.1 at step 43.0
           Precisio